In [1]:
# %%
# ======================
# IMPORT
# ======================
import os  
import torch
from torch.utils.data import DataLoader
from collections import Counter

from src.preprocessing.precomputed import PrecomputedDataset
from src.models.crnn import CRNN
from src.utils import set_seed, evaluate


In [2]:
# %%
# ======================
# CONFIG
# ======================
set_seed(42)

EPOCHS = 80
BATCH_SIZE = 64
LR = 1e-3
WEIGHT_DECAY = 1e-4

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


In [3]:
# %%
# ======================
# DATASET
# ======================
DATA_PATH = "./scripts/data_precomputed"

train_ds = PrecomputedDataset(os.path.join(DATA_PATH, "train"))
val_ds   = PrecomputedDataset(os.path.join(DATA_PATH, "val"))
test_ds  = PrecomputedDataset(os.path.join(DATA_PATH, "test"))

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(
    f"Samples -> train {len(train_ds)} | "
    f"val {len(val_ds)} | test {len(test_ds)}"
)


Samples -> train 4320 | val 120 | test 240


In [4]:
# %%
# ======================
# CLASS WEIGHTS
# ======================
# Estraiamo le etichette da tutto il dataset di train pre-calcolato
print("Conteggio classi in corso (può richiedere qualche secondo)...")
train_labels = [y.item() for _, y in train_ds]
counts = Counter(train_labels)

# Calcoliamo i pesi: 1.0 / frequenza della classe
# Questo serve a bilanciare l'apprendimento se alcune emozioni sono più frequenti
weights = torch.tensor(
    [1.0 / counts[i] for i in range(8)], 
    dtype=torch.float, 
    device=device
)

# Normalizzazione: facciamo in modo che la media dei pesi sia 1
weights = weights / weights.sum() * 8

print("Distribuzione classi nel Train (originali + augmentation):", counts)
print("Pesi applicati alla Loss Function:", weights.detach().cpu().numpy())

Conteggio classi in corso (può richiedere qualche secondo)...
Distribuzione classi nel Train (originali + augmentation): Counter({3: 576, 1: 576, 2: 576, 4: 576, 5: 576, 6: 576, 7: 576, 0: 288})
Pesi applicati alla Loss Function: [1.7777778 0.8888889 0.8888889 0.8888889 0.8888889 0.8888889 0.8888889
 0.8888889]


In [5]:
# %%
# ======================
# MODEL
# ======================
model = CRNN(n_classes=8, n_mels=64).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    patience=6,
    factor=0.5
)

best_val_acc = 0.0


In [6]:
# %%
# ======================
# TRAIN LOOP
# ======================
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    running_correct = 0  # <--- Aggiunto per calcolare l'accuratezza
    total = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)

        loss = torch.nn.functional.cross_entropy(
            logits,
            y,
            weight=weights,
            label_smoothing=0.1
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        # Statistiche
        running_loss += loss.item() * y.size(0)
        # Calcolo predizioni corrette
        preds = logits.argmax(dim=1)
        running_correct += (preds == y).sum().item()
        total += y.size(0)

    train_loss = running_loss / total
    train_acc = running_correct / total  # <--- Calcolo finale accuratezza train

    # Valutazione su validation e test
    val_loss, val_acc = evaluate(model, val_loader, device)
    test_loss, test_acc = evaluate(model, test_loader, device)

    scheduler.step(val_acc)

    # Output nel formato richiesto
    print(
        f"Epoch {epoch:2d}/{EPOCHS} | "
        f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} | "
        f"test acc {test_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_precomputed.pt")

RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "c:\Users\eleon\mnist_env_gpu_11\Lib\site-packages\torch\utils\data\_utils\worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\eleon\mnist_env_gpu_11\Lib\site-packages\torch\utils\data\_utils\fetch.py", line 55, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\eleon\mnist_env_gpu_11\Lib\site-packages\torch\utils\data\_utils\collate.py", line 398, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\eleon\mnist_env_gpu_11\Lib\site-packages\torch\utils\data\_utils\collate.py", line 211, in collate
    return [
           ^
  File "c:\Users\eleon\mnist_env_gpu_11\Lib\site-packages\torch\utils\data\_utils\collate.py", li

In [ ]:
# %%
# ======================
# FINAL RESULT
# ======================
print("BEST VAL ACC:", best_val_acc)
